# Lesson 11 · Convolutional networks (CNNs): a 3×3 filter slides across handwritten digits

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/11-cnn-en.ipynb)

This notebook accompanies [Lesson 11 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/cnn/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/11-cnn.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
The MLP of the earlier lessons connects each neuron to all 784 pixels and has no idea which two pixels sit next to
each other. We write convolution ourselves in numpy, run the demo's very own CNN (8 filters of size 3×3 → ReLU →
max pooling → dense layer) on 10,000 real MNIST images, and check cell by cell how far “when the object moves, the
map moves with it” holds.

**What you'll do:**
1. Write convolution as a loop over one digit, then vectorize it into a single matrix multiplication $Z = PW + b$.
2. Use a vertical-edge and a horizontal-edge filter, then read the 8 filters that the demo's CNN learned by itself.
3. Run the demo's CNN on the test set, count its parameters and multiply-accumulate operations against the MLP's,
   then train a CNN yourself.
4. Check equivariance cell by cell, then shift all 10,000 images to see that the CNN is not yet invariant.
5. Shuffle the pixel positions, and see that the MLP doesn't notice at all while the CNN loses its advantage; then
   look at the receptive field, on the way to Lesson 12.

**Background:** [Lesson 03 · Multilayer networks & the XOR problem](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/)
(hidden layers, $z = xW + b$) and [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/).
**Run time:** under a minute on a CPU, including the first download of MNIST (~11 MB) and of the demo's results file.

Every number in the notebook is computed by the cells below or read from the very results file the demo uses,
and matches the numbers in the demo.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, ScalarFormatter
from numpy.lib.stride_tricks import sliding_window_view

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
SEQUENTIAL = LinearSegmentedColormap.from_list("white_blue", ["#ffffff", BLUE])  # 0 → large
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}"  # never prints “−0”
    return s.replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def sci(x):
    """A very small number, in scientific notation: 2.4e−08."""
    return f"{x:.1e}".replace("-", "−")


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every linear axis (colorbars included) in the English format; call right before
    plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(tick))


print(num(13610, 0), num(0.1, 1), num(-4.3805, 4), pct(0.9798), sci(2.4e-8))

## 2 · The data

MNIST (LeCun et al., 1998) has 70,000 grayscale 28×28 images of handwritten digits: 60,000 training images and
10,000 test images. The cell below downloads the 4 original files from a public mirror and checks their MD5 hashes,
so everyone who runs it gets exactly the same data.

The cell below also downloads `mnist-cnn.json`, the file the lesson's demo reads. It is produced by the course's
program `reel/mnist_mlp/cnn.py`, which trains the CNN 3 times (seeds 0, 1, 2) and a few comparison networks, then
tests each network on shifted test images. All of that takes much longer than a notebook. The file contains **the
weights of the seed-0 CNN** (exactly the network the demo runs in the browser) and the measured accuracies. We will
rerun that network ourselves, and load ready-made only the results that need long training.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of a long training run (the same file as the demo on the web).
# build.py replaces COURSE_DATA_URL with the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
demo = fetch_json("mnist-cnn.json", "d4c7c4c978f9abf403a2193fd907dcd7")
n_tr, n_te = len(train_lab), len(test_lab)
print(
    f"{num(n_tr, 0)} training images + {num(n_te, 0)} test images = {num(n_tr + n_te, 0)};  each image",
    train_img.shape[1:],
)
print("in mnist-cnn.json:", ", ".join(demo))

We divide every pixel by 255 to get numbers in $[0, 1]$: 0 is the background, 1 the darkest stroke. Unlike in the
earlier lessons, the images keep their **28×28 grid** shape instead of being flattened into vectors of 784 numbers,
because convolution needs to know which pixel sits next to which.

In [ ]:
Xtr, ytr = train_img / 255.0, train_lab.astype(int)  # 60,000 × 28 × 28
Xte, yte = test_img / 255.0, test_lab.astype(int)  # 10,000 × 28 × 28
SAMPLES = [s["mnist"] for s in demo["samples"]]  # the 10 test images the demo lets you choose from, one per digit
print("the demo's sample images (indices in the test set):", SAMPLES)
print("their labels:", [int(yte[i]) for i in SAMPLES])
ink_s = test_img[SAMPLES] > 0
margin = min(
    min(np.argmax(ink_s.any(axis=a), axis=1).min(), np.argmax(ink_s.any(axis=a)[:, ::-1], axis=1).min()) for a in (1, 2)
)
print(f"smallest empty margin of the 10 sample images: {margin} pixels")
assert margin >= 4

The figure below shows those 10 images, in the demo's order (one per digit, each with an empty margin of at least
4 pixels on every side, so shifting by up to 4 pixels cuts off no stroke). Each title is the index in the test set
and the true label.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.8))
for ax, i in zip(axes, SAMPLES):
    ax.imshow(Xte[i], cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"#{i}\ndigit {yte[i]}", fontsize=9)  # two lines: on one line, neighboring titles touch
    ax.axis("off")
plt.show()

## 3 · An image is a matrix of numbers; a 3×3 filter slides over it

To a computer, the digit 4 (#109) is just a 28×28 matrix. The cell below prints a corner of it as raw bytes 0–255
(0 is the background, 255 the darkest ink).

In [ ]:
I = Xte[109]  # the digit 4 among the demo's 10 sample images
print("matrix", I.shape, "; rows 8–14, columns 4–13 (raw bytes):")
print(test_img[109][8:15, 4:14])

A **filter** (kernel) $K$ is a small $k \times k$ matrix, here $3 \times 3$. Place it on a $3 \times 3$ window of the
image, multiply each pair of overlapping elements, then add up the 9 products (together with a bias $b$): that is one
cell of the **feature map**. Slide the window one pixel at a time across the whole image, and you get the whole map:

$$z_{ij} = \sum_{m=0}^{k-1}\sum_{n=0}^{k-1} I_{i+m,\,j+n}\, K_{mn} + b .$$

The window must fit entirely inside the image (no zero padding at the border, stride 1), so $i, j$ run from 0 to
$28 - 3 = 25$: the map has $26 \times 26 = 676$ cells. We try the **vertical-edge Sobel** filter, right column minus
left column:

$$K_\text{vertical} = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}.$$

The cell below computes one cell by hand, at the window whose top row is row 12 and whose leftmost column is column 6.

In [ ]:
K_V = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)  # vertical-edge Sobel: right column minus left column
K_H = K_V.T.copy()  # horizontal-edge Sobel: bottom row minus top row
i0, j0 = 12, 6
window = I[i0 : i0 + 3, j0 : j0 + 3]
print("3×3 window of the image:\n", np.round(window, 2))
print("the 9 products I·K:\n", np.round(window * K_V, 2))
print(f"sum of the 9 products = z({i0}, {j0}) = {num((window * K_V).sum(), 4)}")

The window's left column is almost all background and its right column is ink, so right minus left gives a large
positive number: this window sits on the **left edge** of a vertical stroke. Now write the whole sliding as a loop,
counting every multiply-accumulate (MAC) operation:

In [ ]:
def conv2d_loop(I, K, b=0.0):
    """A 'valid' convolution, stride 1, written straight from the formula. Returns the map and the number of
    multiply-accumulates."""
    k = K.shape[0]
    rows, cols = I.shape[0] - k + 1, I.shape[1] - k + 1
    Z, macs = np.zeros((rows, cols)), 0
    for i in range(rows):
        for j in range(cols):
            s = b
            for m in range(k):
                for n in range(k):
                    s += I[i + m, j + n] * K[m, n]  # one multiply-accumulate
                    macs += 1
            Z[i, j] = s
    return Z, macs


t0 = time.perf_counter()
Z_loop, macs = conv2d_loop(I, K_V)
t_loop = time.perf_counter() - t0
print(f"map {Z_loop.shape}, {num(macs, 0)} MACs; z({i0}, {j0}) = {num(Z_loop[i0, j0], 4)}")
peak = tuple(int(v) for v in np.unravel_index(Z_loop.argmax(), Z_loop.shape))
print(f"largest {num(Z_loop.max(), 4)} at cell {peak}, smallest {num(Z_loop.min(), 4)}")
assert macs == 676 * 9 == 6084

### Vectorizing: each window is a row vector

Flatten each $3 \times 3$ window into a row vector of 9 numbers. Stack the 676 windows on top of one another, and you
get a $676 \times 9$ matrix $P$ (this technique is called **im2col**). The filter flattens into a column of 9 numbers,
and the whole map is **a single matrix multiplication**, exactly the course's convention $z = xW + b$, except that
each “image” $x$ is now a window of 9 pixels:

$$Z = P\,W + b, \qquad P \in \mathbb{R}^{676 \times 9},\; W \in \mathbb{R}^{9 \times F}.$$

With $F$ filters, $W$ has $F$ columns and we get $F$ maps at once. numpy's `sliding_window_view` builds $P$ without
copying any data (each pixel appears in up to 9 windows). The function below takes one image or a whole batch of
images.

In [ ]:
def patches(X, k=3):
    """Every k×k window of the image (or batch of images) X, flattened into row vectors: (..., H−k+1, W−k+1, k·k)."""
    *lead, H, W = X.shape
    return sliding_window_view(X, (k, k), axis=(-2, -1)).reshape(*lead, H - k + 1, W - k + 1, k * k)


def conv2d(X, W, b=0.0):
    """Convolution with F filters: W is k·k × F (row m·k + n ↔ element K[m, n]); returns (..., H−k+1, W−k+1, F)."""
    return patches(X, int(np.sqrt(W.shape[0]))) @ W + b


P1 = patches(I)
t0 = time.perf_counter()
Z_vec = conv2d(I, K_V.reshape(9, 1))[..., 0]
t_vec = time.perf_counter() - t0
print("P of one image:", P1.reshape(-1, 9).shape, " ·  row of the window (12, 6):", np.round(P1[i0, j0], 2))
print(
    f"loop {num(t_loop * 1000, 1)} ms, matrix multiplication {num(t_vec * 1000, 3)} ms; largest difference {sci(np.abs(Z_vec - Z_loop).max())}"
)
assert np.allclose(Z_vec, Z_loop, atol=1e-12)

### “Convolution” in deep learning is really cross-correlation

In mathematics and signal processing, convolution **flips** the filter before sliding it:
$\sum_{m,n} I_{i+m,j+n} K_{k-1-m,\,k-1-n}$. The formula above doesn't flip, so its proper name is
**cross-correlation**. Deep learning libraries all compute cross-correlation but still call it convolution. With a
learned filter, the flip doesn't matter: a network that used true convolution would just learn the same filter,
flipped. numpy has both operations for 1-D signals, so we can check:

In [ ]:
row = I[14]  # row 14 of the digit 4
k1 = np.array([1.0, 2.0, -3.0])  # an asymmetric 1-D filter
same = np.allclose(np.convolve(row, k1, "valid"), np.correlate(row, k1[::-1], "valid"))
print("np.convolve(x, k) = np.correlate(x, k flipped):", same)
print("vertical Sobel flipped in both directions = −vertical Sobel:", np.array_equal(K_V[::-1, ::-1], -K_V))
assert same

So with “true” convolution, the map of the vertical Sobel filter would only change sign. From here on, we follow the
common habit and call it convolution.

The figure below: on the left, the digit 4 with the $3 \times 3$ window we just computed (green frame); on the right,
the whole $26 \times 26$ map of the vertical-edge filter (blue is positive, orange negative, white 0), with the cell
we just computed marked by a green frame. Cell $(i, j)$ of the map corresponds to the window whose top-left corner is
at $(i, j)$, i.e. whose center is at pixel $(i+1, j+1)$.

In [ ]:
def show_map(ax, Z, title, vmax=None):
    vmax = vmax or np.abs(Z).max() or 1
    im = ax.imshow(Z, cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im


fig, (a, b) = plt.subplots(1, 2, figsize=(7.4, 3.6))
a.imshow(I, cmap="gray_r", vmin=0, vmax=1)
a.add_patch(Rectangle((j0 - 0.5, i0 - 0.5), 3, 3, fill=False, ec=GREEN, lw=2.5))
a.set_title("28×28 image, window at (12, 6)", fontsize=10)
a.axis("off")
im = show_map(b, Z_vec, "26×26 vertical-edge map")
b.add_patch(Rectangle((j0 - 0.5, i0 - 0.5), 1, 1, fill=False, ec=GREEN, lw=2.5))
fig.colorbar(im, ax=b, fraction=0.046)
fmt_axes(fig)
plt.show()

## 4 · Edge-detecting filters

The vertical-edge filter gives positive numbers at the **left edge** of each vertical stroke (the window goes from
background 0 to ink 1) and negative numbers at the right edge (from ink back to background); in the middle of a flat
region, the right and left columns are equal, so it gives 0. Swap the roles of rows and columns and you get the
**horizontal-edge Sobel** filter (bottom row minus top row): positive at the top edge of a horizontal stroke,
negative at the bottom edge.

The two filters are the two columns of $W$ (size $9 \times 2$), so one matrix multiplication gives both maps. The
figure below draws the digit 4 and the two maps on the same color scale (blue: positive, orange: negative).

In [ ]:
W_sobel = np.stack([K_V.ravel(), K_H.ravel()], axis=1)  # 9 × 2: one filter per column
Z2 = conv2d(I, W_sobel)  # 26 × 26 × 2
vmax = np.abs(Z2).max()
fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.4))
axes[0].imshow(I, cmap="gray_r", vmin=0, vmax=1)
axes[0].set_title("digit 4 (#109)", fontsize=10)
axes[0].axis("off")
show_map(axes[1], Z2[..., 0], "vertical-edge Sobel", vmax)
im = show_map(axes[2], Z2[..., 1], "horizontal-edge Sobel", vmax)
fig.colorbar(im, ax=axes[2], fraction=0.046)
fmt_axes(fig)
plt.show()
for name, Z in zip(["vertical", "horizontal"], np.moveaxis(Z2, -1, 0)):
    print(
        f"{name:>10} edges: largest {num(Z.max(), 2)}, smallest {num(Z.min(), 2)}, {(np.abs(Z) > 1).sum()} cells with |z| > 1"
    )

Only the windows that sit on the edge of a stroke give a large $|z|$; the white background and the inside of the
strokes both give 0. The vertical-edge map lights up along the two upright strokes of the 4 (the slanted stroke on
the left, the vertical stroke on the right), the horizontal-edge map along the crossbar. A $3 \times 3$ filter is a
**local pattern**, and the map tells you where that pattern appears in the image.

## 5 · Many filters → many maps: the demo's CNN

The CNN in the demo doesn't use hand-designed filters. It has $F = 8$ filters of size $3 \times 3$ with random initial
weights, and **learns** them together with the rest of the network by backpropagation (Lesson 06), with exactly the
MLP's recipe: SGD, $\eta = 0.1$, batches of 64 images, 5 epochs over 60,000 images. The cell below reads the seed-0
weights from the demo's file (the dense layer is stored as base64-encoded float32) and prints the size of each part.

In [ ]:
CNN = {
    "Wc": np.array(demo["conv"]["W"]),  # 9 × 8: each column is a flattened 3×3 filter
    "bc": np.array(demo["conv"]["b"]),  # 8 biases, one per filter
    "Wd": np.frombuffer(base64.b64decode(demo["dense"]["W"]), "<f4").astype(float).reshape(-1, 10),  # 1,352 × 10
    "bd": np.array(demo["dense"]["b"]),
}
for name, v in CNN.items():
    print(f"{name}: {v.shape}")
F = CNN["Wc"].shape[1]
assert (demo["k"], F) == (3, 8)

The figure below: the top row is the 8 learned filters (each 3×3 square is one weight; blue positive, orange
negative), the bottom row is the 8 feature maps of the digit 4 after ReLU (white is 0; the darker the blue, the
larger). Above each filter is its similarity (cosine) with the vertical Sobel filter (“v”) and the horizontal Sobel
filter (“h”): ±1 means identical (or of opposite sign), 0 means unrelated.

In [ ]:
def cos(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))


maps = np.maximum(conv2d(I, CNN["Wc"], CNN["bc"]), 0)  # 26 × 26 × 8, after ReLU
fig, axes = plt.subplots(2, F, figsize=(10, 3.2))
for f in range(F):
    w = CNN["Wc"][:, f]
    show_map(
        axes[0, f], w.reshape(3, 3), f"#{f + 1}\nv {num(cos(w, K_V.ravel()), 2)} · h {num(cos(w, K_H.ravel()), 2)}"
    )
    axes[1, f].imshow(maps[..., f], cmap=SEQUENTIAL, vmin=0)
    axes[1, f].axis("off")
plt.tight_layout()
plt.show()
for f in range(F):
    w = CNN["Wc"][:, f]
    print(
        f"filter #{f + 1}: b = {num(CNN['bc'][f], 3):>6}, cos with vertical Sobel {num(cos(w, K_V.ravel()), 2):>5}, "
        f"horizontal {num(cos(w, K_H.ravel()), 2):>5}"
    )

Nobody designed these 8 filters, yet a few of them look a lot like edge detectors: #4 is almost the vertical Sobel
filter (cosine 0.93), #6 is the horizontal Sobel filter with the opposite sign (−0.87), #1 responds to diagonal edges.
Filter #5 has a positive bias, so it lights up the background too; the filters with a negative $b$ light up only
where there is a stroke.

Where did they start from? `cnn.py` initializes the filters with the He distribution, $\mathcal{N}(0, 2/9)$, from
seed 0; that is the first random draw, so we can rebuild exactly the 8 initial filters and compare them with the
8 filters after 5 epochs of learning.

In [ ]:
Wc0 = np.random.default_rng(0).normal(0, np.sqrt(2 / 9), (9, F))  # cnn.py's first draw with seed 0


def edge_like(w):
    """The largest |cosine| with the vertical or the horizontal Sobel filter: 1 is a perfect edge detector (opposite
    sign included)."""
    return max(abs(cos(w, K_V.ravel())), abs(cos(w, K_H.ravel())))


for f in range(F):
    w0, w1 = Wc0[:, f], CNN["Wc"][:, f]
    print(
        f"#{f + 1}: cos(initial, learned) = {num(cos(w0, w1), 2)};  edge-likeness {num(edge_like(w0), 2)} → {num(edge_like(w1), 2)};"
        f"  length {num(np.linalg.norm(w0), 2)} → {num(np.linalg.norm(w1), 2)}"
    )
n0, n1 = (sum(edge_like(W[:, f]) >= 0.75 for f in range(F)) for W in (Wc0, CNN["Wc"]))
grow = np.linalg.norm(CNN["Wc"], axis=0) / np.linalg.norm(Wc0, axis=0)
turn = np.degrees(np.arccos([cos(Wc0[:, f], CNN["Wc"][:, f]) for f in range(F)]))  # angle between initial and learned
print(f"filters that look like edge detectors (|cos| ≥ 0.75): {n0}/8 at initialization, {n1}/8 after learning")
print(
    f"the length grows by a factor of {num(grow.min(), 2)} to {num(grow.max(), 2)} (median {num(np.median(grow), 2)})"
)
print(f"angle to the initialization: {num(turn.min(), 0)}° to {num(turn.max(), 0)}° (the most: #{turn.argmax() + 1})")

A surprise: each learned filter stays **close to its own direction at initialization** (cosine 0.70 to 0.95, that
is, 18° to 45° apart; #6 turned the most). Learning mostly stretches them (their length grows by a factor of 1.05 to
2.62) and turns them toward edge detectors: the number of filters with $|\cos| \ge 0.75$ against a Sobel filter grows
from 1 to 4. With one convolutional layer and 9 weights, a random filter is already a usable detector of local
patterns; section 12 will measure how much learning the filters helps.

## 6 · The whole network: ReLU → max pooling → dense layer

After the convolution, the demo's network does three things:
1. **ReLU** on each cell of the 8 maps of size $26 \times 26$.
2. **2×2 max pooling**: split each map into non-overlapping $2 \times 2$ blocks and keep the largest number of each
   block, so $26 \times 26 \times 8$ becomes $13 \times 13 \times 8$. Pooling is the main topic of Lesson 12; here
   you only need to know that it makes the maps 4 times smaller.
3. Flatten the $13 \cdot 13 \cdot 8 = 1{,}352$ numbers into a row vector $h$ and compute **10 logits**
   $z = hW_d + b_d$ with a dense layer, as in every earlier lesson.

The cell below writes the whole network (it runs in batches of 200 images: small intermediate arrays are faster),
prints the size of the data after each step for one image, then runs it on all 10,000 test images.

In [ ]:
def maxpool2(A):
    """2×2 max pooling, stride 2, over the two spatial axes of (n, H, W, F)."""
    return np.maximum(np.maximum(A[:, 0::2, 0::2], A[:, 0::2, 1::2]), np.maximum(A[:, 1::2, 0::2], A[:, 1::2, 1::2]))


def cnn_logits(P, X, chunk=200, verbose=False):
    out = []
    for i in range(0, len(X), chunk):
        A = np.maximum(conv2d(X[i : i + chunk], P["Wc"], P["bc"]), 0)  # convolution + ReLU: n × 26 × 26 × 8
        H = maxpool2(A)  # n × 13 × 13 × 8
        h = H.reshape(len(H), -1)  # n × 1,352, in (row, column, filter) order, as in cnn.py
        out.append(h @ P["Wd"] + P["bd"])
        if verbose:
            print(
                "image",
                X[i : i + chunk].shape,
                "→ maps",
                A.shape,
                "→ pooling",
                H.shape,
                "→",
                h.shape,
                "→ logits",
                out[-1].shape,
            )
    return np.concatenate(out)


z109 = cnn_logits(CNN, Xte[109:110], verbose=True)[0]
print(f"image #109 (digit {yte[109]}): the CNN predicts {z109.argmax()}")
Z_cnn = cnn_logits(CNN, Xte)
acc_cnn = (Z_cnn.argmax(1) == yte).mean()
SHIFTS = [(dx, dy) for dy in range(-4, 5) for dx in range(-4, 5)]  # the order of the file's 9 × 9 grid of shifts
CENTER = SHIFTS.index((0, 0))
print(f"CNN: {num((Z_cnn.argmax(1) == yte).sum(), 0)}/10,000 test images correct = {pct(acc_cnn)}")
print(
    f"the demo's file says: {pct(demo['grid']['cnn'][CENTER])}; the MLP of the earlier lessons: {pct(demo['grid']['mlp'][CENTER])}"
)
assert abs(acc_cnn - demo["grid"]["cnn"][CENTER]) < 5e-4, "differs from the demo: check the data"
assert all(
    Z_cnn[i].argmax() == yte[i] for i in SAMPLES
)  # the demo picked 10 images that both the CNN and the MLP get right

The network running in the notebook gets exactly 97.98%, as in the demo (same weights, same computation), higher
than the 784 → 128 → 64 → 10 MLP of the earlier lessons (97.27%), even though it learns with the same recipe: SGD,
$\eta = 0.1$, batches of 64, 5 epochs.

### 🤔 Predict first
The convolutional layer has 72 weights (8 filters × 9), the dense layer 13,520 ($1{,}352 \times 10$). For one image,
which layer costs more **multiply-accumulate operations**? And does the whole CNN need fewer operations than the MLP,
given that it has about 8 times fewer parameters?

In [ ]:
SIZES = demo["mlpSizes"]  # [784, 128, 64, 10]: the MLP of the earlier lessons
mlp_params = sum((n_in + 1) * n_out for n_in, n_out in itertools.pairwise(SIZES))
mlp_macs = sum(n_in * n_out for n_in, n_out in itertools.pairwise(SIZES))
M, Q = 28 - 3 + 1, (28 - 3 + 1) // 2  # 26: side of a map; 13: after pooling
conv_params, conv_macs = (9 + 1) * F, M * M * 9 * F  # each cell of each map: 9 multiply-accumulates
dense_params, dense_macs = (Q * Q * F + 1) * 10, Q * Q * F * 10
print(
    f"convolution: {num(conv_params, 0):>7} parameters, {num(conv_macs, 0):>7} MACs  (72 weights reused at {M * M} positions)"
)
print(f"dense      : {num(dense_params, 0):>7} parameters, {num(dense_macs, 0):>7} MACs")
print(f"CNN        : {num(conv_params + dense_params, 0):>7} parameters, {num(conv_macs + dense_macs, 0):>7} MACs")
print(f"MLP        : {num(mlp_params, 0):>7} parameters, {num(mlp_macs, 0):>7} MACs")
print(
    f"the CNN has {num(mlp_params / (conv_params + dense_params), 2)} times fewer parameters and {num(mlp_macs / (conv_macs + dense_macs), 2)} times fewer MACs;"
    f" the convolution costs {num(conv_macs / dense_macs, 2)} times as many MACs as the dense layer"
)
assert (conv_params, conv_macs, dense_params, dense_macs) == (80, 48672, 13530, 13520)
assert (mlp_params, mlp_macs) == (109386, 109184)

The convolutional layer has only 80 parameters (72 weights + 8 biases) but costs **48,672** multiply-accumulates per
image, 3.6 times as many as the dense layer with its 13,530 parameters. Each convolution weight is reused at
676 positions, so for convolution, the number of parameters and the number of operations are two different things.
The whole CNN is still cheaper than the MLP: 8 times fewer parameters, 1.76 times fewer operations.

How much does weight sharing save? If each position had its own set of weights (a “locally connected” layer),
the 8 maps would need $676 \cdot 8 \cdot (9 + 1)$ parameters. With large images, the difference is even bigger: the
first layer of AlexNet (227×227×3 images, 96 filters of 11×11×3, stride 4, 55×55 maps) is the example used in the
CS231n course.

In [ ]:
print(f"8 maps without weight sharing: {num(M * M * F * (9 + 1), 0)} parameters, instead of {conv_params}")
side = (227 - 11) // 4 + 1  # CS231n's formula (W − F)/S + 1
alex_local = side * side * 96 * (11 * 11 * 3 + 1)
alex_shared = 96 * (11 * 11 * 3 + 1)
print(
    f"AlexNet layer 1: {side}×{side}×96 maps; without sharing {num(alex_local, 0)} parameters, with sharing {num(alex_shared, 0)}"
)
print(
    f"a 224×224 color image has {num(224 * 224 * 3, 0)} values; a dense neuron connected to all of them needs as many weights"
)

## 7 · Learning the filters: the gradient of a shared weight

The filters are learned by backpropagation like any other weight (Lesson 06). Since $Z = PW + b$ is just a matrix
multiplication, the gradient also has the familiar form: given $\partial\mathcal{L}/\partial Z$,

$$\frac{\partial \mathcal{L}}{\partial W} = P^\top \frac{\partial \mathcal{L}}{\partial Z}, \qquad
  \frac{\partial \mathcal{L}}{\partial b} = \sum_\text{all positions} \frac{\partial \mathcal{L}}{\partial Z}.$$

$P^\top$ has 676 columns for each image, so the gradient of each filter weight is **the sum of the contributions of
all 676 positions** (and of every image in the batch): a weight that is shared gets a gradient that is summed. Through
max pooling, the gradient flows back only to the largest cell of each $2 \times 2$ block; through ReLU, only to the
positive cells. The function below follows exactly the steps of `cnn.py`: He initialization ($\mathcal{N}(0, 2/9)$
for the filters, $\mathcal{N}(0, 1/1{,}352)$ for the dense layer), SGD, batches of 64.

In [ ]:
CORNERS = [(0, 0), (0, 1), (1, 0), (1, 1)]  # the 4 cells of a 2×2 block


def init_cnn(rng, F=8):
    return {
        "Wc": rng.normal(0, np.sqrt(2 / 9), (9, F)),
        "bc": np.zeros(F),
        "Wd": rng.normal(0, np.sqrt(1 / (Q * Q * F)), (Q * Q * F, 10)),
        "bd": np.zeros(10),
    }


def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


def cnn_grads(P, x, y, learn_filters=True):
    """Forward then backward pass for one batch; returns the gradient of the mean cross-entropy with respect to every
    parameter, and the loss."""
    pt = patches(x)  # n × 26 × 26 × 9
    A = pt @ P["Wc"] + P["bc"]
    R = np.maximum(A, 0)
    h = maxpool2(R).reshape(len(x), -1)
    dz = softmax(h @ P["Wd"] + P["bd"])
    loss = float(-np.log(dz[np.arange(len(y)), y]).mean())
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    g = {"Wd": h.T @ dz, "bd": dz.sum(0)}
    if learn_filters:
        C = np.stack([R[:, r::2, c::2] for r, c in CORNERS], axis=-1)  # n × 13 × 13 × F × 4
        arg = C.argmax(-1)  # which cell wins in each block (a tie goes to the first cell, as in cnn.py)
        dH = (dz @ P["Wd"].T).reshape(arg.shape)
        dA = np.zeros_like(A)
        for k, (r, c) in enumerate(CORNERS):
            dA[:, r::2, c::2] = dH * (arg == k)  # the gradient flows back only to the block's winning cell
        dA *= A > 0  # and only through the cells whose ReLU is on
        g["Wc"] = pt.reshape(-1, 9).T @ dA.reshape(-1, P["Wc"].shape[1])  # Pᵀ · ∂𝓛/∂Z: a sum over every position
        g["bc"] = dA.reshape(-1, P["Wc"].shape[1]).sum(0)
    return g, loss

Before trusting the formula, we check it with **finite differences**, as in Lesson 06: nudge each weight by a small
$\pm\varepsilon$, see how much the loss changes, then compare with the gradient we just computed. The cell below does
that for all 9 weights of filter #1 and 5 weights of the dense layer, on a batch of 16 images with a freshly
initialized network.

In [ ]:
def loss_of(P, x, y):
    return cnn_grads(P, x, y, learn_filters=False)[1]


P_chk = init_cnn(np.random.default_rng(7))
x_chk, y_chk = Xtr[:16], ytr[:16]
g_chk, _ = cnn_grads(P_chk, x_chk, y_chk)
eps, analytic, numeric = 1e-6, [], []
for name, idx in [("Wc", (t, 0)) for t in range(9)] + [("Wd", (r, 3)) for r in (0, 200, 700, 1000, 1351)]:
    Pp = {k: v.copy() for k, v in P_chk.items()}
    Pm = {k: v.copy() for k, v in P_chk.items()}
    Pp[name][idx] += eps
    Pm[name][idx] -= eps
    numeric.append((loss_of(Pp, x_chk, y_chk) - loss_of(Pm, x_chk, y_chk)) / (2 * eps))
    analytic.append(g_chk[name][idx])
analytic, numeric = np.array(analytic), np.array(numeric)
rel = np.linalg.norm(analytic - numeric) / np.linalg.norm(analytic + numeric)
print("∂𝓛/∂W of filter #1 (×1,000):", " ".join(num(1000 * v, 2) for v in g_chk["Wc"][:, 0]))
print(f"relative error against finite differences (14 weights): {sci(rel)}")
assert rel < 1e-6

The gradient matches the finite differences. Now for real training. The demo's recipe (5 epochs over 60,000 images)
is too long for this notebook, so we run **one epoch over the first 20,000 training images** (313 steps), seed 0,
then measure on all 10,000 test images. The function `train_cnn` keeps exactly the order of random draws of
`cnn.py` (initialization, then one permutation per epoch).

In [ ]:
def train_cnn(seed=0, n=20_000, epochs=1, X=Xtr, y=ytr, eta=0.1, learn_filters=True):
    """SGD with batches of 64, as in reel/mnist_mlp/cnn.py. learn_filters=False: keep the random filters fixed and learn
    only the dense layer."""
    rng = np.random.default_rng(seed)
    P = init_cnn(rng)
    X, y = X[:n], y[:n]
    losses = []
    for _ in range(epochs):
        order = rng.permutation(len(X))
        for i in range(0, len(order), 64):
            idx = order[i : i + 64]
            g, loss = cnn_grads(P, X[idx], y[idx], learn_filters)
            for k in g:
                P[k] -= eta * g[k]
            losses.append(loss)
    return P, losses


def accuracy(P, X=Xte, y=yte):
    return float((cnn_logits(P, X).argmax(1) == y).mean())


t0 = time.perf_counter()
P_live, loss_live = train_cnn(seed=0)
acc_live = accuracy(P_live)
print(
    f"{len(loss_live)} steps over {num(20_000, 0)} images ({num(time.perf_counter() - t0, 1)} seconds); {pct(acc_live)} correct on 10,000 test images"
)
full_steps = 5 * len(range(0, 60_000, 64))
print(f"the demo's full recipe: {num(full_steps, 0)} steps, {num(full_steps / len(loss_live), 1)} times as many")
print(f"loss: first 25 steps {num(np.mean(loss_live[:25]), 3)}, last 25 steps {num(np.mean(loss_live[-25:]), 3)}")
assert len(loss_live) == 313 and abs(acc_live - 0.9266) < 0.003  # measured when the notebook was written (aarch64)

Only 313 steps over a third of the training set, and the CNN already gets 92.66% right. The demo's full recipe is
15 times longer (4,690 steps). The cell below runs it if you set `FULL = True` (it also takes about 15 times as long
as the cell above). When writing the notebook, we ran it and got exactly the weights in the demo's file, bit for bit
after rounding to float32 as `cnn.py` does; on another machine, the additions round differently, so the last digit
may differ.

In [ ]:
FULL = False  # True: 5 epochs × 60,000 images, exactly the recipe that produced the demo's network
if FULL:
    P_full, _ = train_cnn(seed=0, n=60_000, epochs=5)
    P32 = {k: v.astype(np.float32).astype(float) for k, v in P_full.items()}  # the demo stores its weights in float32
    print("largest difference from the demo's weights:", max(float(np.abs(P32[k] - CNN[k]).max()) for k in CNN))
    print("correct on the test set:", pct(accuracy(P32)))

## 8 · The opponent: the MLP of the earlier lessons

For a fair comparison, we need the very 784 → 128 → 64 → 10 MLP that the demo uses. Retraining it takes only a few
seconds: same recipe, same seed 0 as `reel/mnist_mlp/train.py` (Lesson 04 rewrites this function step by step).

In [ ]:
def train_mlp(X, y, seed=0, epochs=5, perm=None):
    """The course's MLP, SGD with batches of 64, η = 0.1. perm: permutes the rows of W₁ at initialization (used in
    section 11)."""
    rng = np.random.default_rng(seed)
    params = [[rng.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]
    if perm is not None:
        params[0][0] = params[0][0][perm]
    for _ in range(epochs):
        order = rng.permutation(len(X))
        for i in range(0, len(order), 64):
            idx = order[i : i + 64]
            x, (h1, h2, z) = X[idx], mlp_forward(params, X[idx])
            dz = softmax(z)
            dz[np.arange(len(idx)), y[idx]] -= 1
            dz /= len(idx)
            dh2 = (dz @ params[2][0].T) * (h2 > 0)
            dh1 = (dh2 @ params[1][0].T) * (h1 > 0)
            for layer, (gW, gb) in zip(
                params, [(x.T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]
            ):
                layer[0] -= 0.1 * gW
                layer[1] -= 0.1 * gb
    return params


def mlp_forward(params, X):
    """X: n × 784 (or n × 28 × 28, which gets flattened). Returns h₁, h₂ and the 10 logits."""
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, X.reshape(len(X), -1) @ W1 + b1)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


t0 = time.perf_counter()
MLP = train_mlp(Xtr.reshape(-1, 784), ytr)
acc_mlp = float((mlp_forward(MLP, Xte)[2].argmax(1) == yte).mean())
print(
    f"MLP: {pct(acc_mlp)} correct on the test set ({num(time.perf_counter() - t0, 1)} seconds); the demo's file says {pct(demo['grid']['mlp'][CENTER])}"
)
assert abs(acc_mlp - 0.9727) < 5e-4, "differs from the demo: check the data"

Same training recipe, same seed: 97.27% correct, as in the demo and in Lesson 04.

## 9 · When the object moves, the map moves with it: equivariance

The function `shift` below moves the stroke `dx` pixels to the right and `dy` pixels down (negative numbers mean
left, up), filling the empty space with 0, exactly as `cnn.py` and the demo do.

In [ ]:
def shift(X, dx, dy):
    """Move the stroke dx pixels to the right, dy pixels down (X: n × 28 × 28); newly exposed pixels are 0."""
    out = np.zeros_like(X)
    src = lambda d: slice(max(0, -d), 28 - max(0, d))
    dst = lambda d: slice(max(0, d), 28 - max(0, -d))
    out[:, dst(dy), dst(dx)] = X[:, src(dy), src(dx)]
    return out


def move_map(Z, dx, dy, fill):
    """Move a map (n × 26 × 26 × F) by exactly (dx, dy) cells; newly exposed cells get the value fill (one number per
    filter)."""
    out = np.broadcast_to(fill, Z.shape).copy()
    H, W = Z.shape[1:3]
    out[:, max(0, dy) : H - max(0, -dy), max(0, dx) : W - max(0, -dx)] = Z[
        :, max(0, -dy) : H - max(0, dy), max(0, -dx) : W - max(0, dx)
    ]
    return out

### 🤔 Predict first
Shift the digit 4 by 3 pixels to the right and 2 pixels down, then recompute the CNN's 8 maps (before ReLU). Compared
with the 8 old maps shifted by exactly (3, 2) cells, how many of the 676 positions match perfectly? And does the CNN
still predict the 4 correctly?

In [ ]:
dx, dy = 3, 2
x0, x1 = Xte[109:110], shift(Xte[109:110], dx, dy)
Z0, Z1 = conv2d(x0, CNN["Wc"], CNN["bc"]), conv2d(x1, CNN["Wc"], CNN["bc"])
want = move_map(Z0, dx, dy, CNN["bc"])  # newly exposed cells see only the 0 background, so they equal exactly b
match = np.isclose(Z1, want, rtol=0, atol=1e-12).all(-1)[0]  # a match in all 8 filters
exact = (Z1 == want).all(-1)[0]
print(f"{match.sum()}/676 positions match to 10⁻¹² in all 8 maps ({exact.sum()} match bit for bit)")
p0, p1 = softmax(cnn_logits(CNN, x0))[0], softmax(cnn_logits(CNN, x1))[0]
print(
    f"CNN: the original image → predicts {p0.argmax()} (p = {pct(p0.max())}); shifted ({dx}, {dy}) → predicts {p1.argmax()} (p = {pct(p1.max())})"
)
assert match.sum() == 676

All 676 positions match: the map of the shifted image **is** the old map, shifted. The reason is right there in the
formula: each cell of the map looks only at the $3 \times 3$ window beneath it, with the same filter, so when the
window moves, the resulting cell moves with it. This property is called **equivariance** to translation (translation
equivariance): shifting then convolving equals convolving then shifting. It is built into the structure, with nothing
to learn, and it holds exactly under these conditions: shifts by whole pixels, stride 1, and away from the image
border (a stroke pushed outside is lost; the 10 sample images all have empty margins ≥ 4 pixels).

But the CNN still changes its answer: shifted by (3, 2), it predicts 2 instead of 4. The equivariance of the
convolutional layer is not yet **invariance** of the whole network, meaning an output that doesn't change when the
object moves. We check all 10 sample images and every shift in $[-4, 4]^2$, then follow the signal through each step
after the convolution.

In [ ]:
moves = [(a, b) for b in range(-4, 5) for a in range(-4, 5) if (a, b) != (0, 0)]
Xs = Xte[SAMPLES]
Zs = conv2d(Xs, CNN["Wc"], CNN["bc"])
n_ok = 0
for a, b in moves:
    ok = np.isclose(conv2d(shift(Xs, a, b), CNN["Wc"], CNN["bc"]), move_map(Zs, a, b, CNN["bc"]), rtol=0, atol=1e-12)
    n_ok += int(ok.all(axis=(1, 2, 3)).sum())  # images that match at all 676 positions × 8 filters
print(f"convolution: equivariant on {n_ok}/{len(moves) * len(SAMPLES)} (sample image, shift) pairs")
assert n_ok == len(moves) * len(SAMPLES)

H0 = maxpool2(np.maximum(Zs, 0))  # 10 × 13 × 13 × 8 after ReLU and pooling
for k in range(1, 5):
    Hk = maxpool2(np.maximum(conv2d(shift(Xs, k, 0), CNN["Wc"], CNN["bc"]), 0))
    if k % 2 == 0:  # even shift: does the map after pooling move by exactly k/2 cells?
        same = np.array_equal(Hk[:, :, k // 2 :], H0[:, :, : 13 - k // 2])
        print(
            f"shift by {k} pixels to the right: does the map after pooling move by exactly {k // 2} cell{'s' * (k > 2)}? {same}"
        )
    else:  # odd shift: compare with the old map moved by 0, 1 and 2 cells (k = 3 lies between 1 and 2 cells)
        best = min(
            np.linalg.norm(Hk[:, :, c:] - H0[:, :, : 13 - c]) / np.linalg.norm(H0[:, :, : 13 - c]) for c in (0, 1, 2)
        )
        print(
            f"shift by {k} pixel{'s' * (k > 1)} to the right: the map after pooling differs from every whole-cell shift of the old one by ≥ {pct(best, 1)}"
        )

The three steps after the convolution react differently to a shift:
- Convolution: equivariant on all 800 pairs (10 sample images × 80 nonzero shifts).
- 2×2 max pooling: shift by an **even** number of pixels, and the small map moves by exactly half as many cells, still
  equivariant. Shift by an **odd** number, and the $2 \times 2$ blocks are cut in different places: the map after
  pooling differs from every whole-cell shift of the old one by at least 52.9%. Pooling can only ignore small shifts
  *within* a block, and not always even then.
- Dense layer: it ties a separate weight to each of the 13 × 13 × 8 cells. A map shifted by one cell feeds the signal
  into completely different weights, so this layer is not equivariant at all.

The figure below shows the first step. The top row is the original digit 4, the bottom row the digit shifted by
(3, 2). The middle column is the map of filter #4 (similar to the vertical Sobel filter), with a green circle at the
largest cell; in the bottom row, the dashed square is the old position of that cell. Right column: at the top, the
old map shifted by (3, 2) cells; at the bottom, the difference between it and the new map, a uniform light gray
(equal to 0) at every cell. The maps' background is light orange because a window of all-0 background gives exactly
$z = b$, and the $b$ of filter #4 is negative.

In [ ]:
f4 = 3  # filter #4
fig, axes = plt.subplots(2, 3, figsize=(9.6, 6.2))
peak0 = np.unravel_index(Z0[0, ..., f4].argmax(), (M, M))
for row, (x, Z, title) in enumerate([(x0, Z0, "original image"), (x1, Z1, f"shifted ({dx}, {dy})")]):
    axes[row, 0].imshow(x[0], cmap="gray_r", vmin=0, vmax=1)
    axes[row, 0].set_title(title, fontsize=10)
    axes[row, 0].axis("off")
    peak = np.unravel_index(Z[0, ..., f4].argmax(), (M, M))
    show_map(
        axes[row, 1], Z[0, ..., f4], f"filter #4: largest {num(Z[0, ..., f4].max(), 2)} at {tuple(map(int, peak))}"
    )
    axes[row, 1].plot(peak[1], peak[0], "o", ms=14, mfc="none", mec=GREEN, mew=2.5)
    if row == 1:
        axes[row, 1].add_patch(Rectangle((peak0[1] - 0.5, peak0[0] - 0.5), 1, 1, fill=False, ec=GREEN, lw=1.5, ls="--"))
diff = Z1[0, ..., f4] - want[0, ..., f4]
show_map(axes[0, 2], move_map(Z0, dx, dy, CNN["bc"])[0, ..., f4], f"old map shifted by ({dx}, {dy}) cells")
show_map(axes[1, 2], diff, f"difference: 0 at {(diff == 0).sum()}/676 cells", vmax=np.abs(Z0[0, ..., f4]).max())
plt.tight_layout()
plt.show()

## 10 · Shifting all 10,000 test images

One image doesn't tell us much. We shift **every** test image by $k$ pixels, $k = 0 \dots 4$, in 4 directions
(left, right, up, down), then average the accuracy over the 4 directions, as the demo's chart does.

Equivariance gives us a shortcut: instead of shifting the images and convolving 17 times, we pad each image with
4 zero pixels on every side (36 × 36), convolve **once** (34 × 34 maps), then cut out the 26 × 26 window that
corresponds to each shift. The cell below checks that this shortcut gives the same result, bit for bit, as the
direct approach, then runs the demo's CNN and the MLP we just trained on the 17 shifts.

In [ ]:
AXIS = [(0, 0)] + [d for k in range(1, 5) for d in [(k, 0), (-k, 0), (0, k), (0, -k)]]  # 17 shifts along the two axes


def dense_head(P, A):
    return maxpool2(np.maximum(A, 0)).reshape(len(A), -1) @ P["Wd"] + P["bd"]


Xpad = np.pad(Xte[:50], ((0, 0), (4, 4), (4, 4)))
Apad = conv2d(Xpad, CNN["Wc"], CNN["bc"])  # 50 × 34 × 34 × 8
assert all(
    np.array_equal(Apad[:, 4 - b : 30 - b, 4 - a : 30 - a], conv2d(shift(Xte[:50], a, b), CNN["Wc"], CNN["bc"]))
    for a, b in AXIS
)

t0 = time.perf_counter()
right = {"cnn": {}, "mlp": {}}  # right/wrong for each test image, for each shift
for i in range(0, len(Xte), 200):
    Apad = conv2d(np.pad(Xte[i : i + 200], ((0, 0), (4, 4), (4, 4))), CNN["Wc"], CNN["bc"])
    for a, b in AXIS:
        ok = dense_head(CNN, Apad[:, 4 - b : 30 - b, 4 - a : 30 - a]).argmax(1) == yte[i : i + 200]
        right["cnn"].setdefault((a, b), []).append(ok)
for a, b in AXIS:
    right["cnn"][(a, b)] = np.concatenate(right["cnn"][(a, b)])
    right["mlp"][(a, b)] = mlp_forward(MLP, shift(Xte, a, b))[2].argmax(1) == yte
axis_acc = {m: [np.mean([right[m][d].mean() for d in AXIS if max(map(abs, d)) == k]) for k in range(5)] for m in right}
print(f"17 shifts × 10,000 images × 2 networks ({num(time.perf_counter() - t0, 1)} seconds)")
for m, name in [("mlp", "MLP"), ("cnn", "CNN seed 0")]:
    print(f"{name:>10}: " + " · ".join(f"k = {k}: {pct(v)}" for k, v in enumerate(axis_acc[m])))
assert np.allclose(axis_acc["cnn"], demo["axis"]["cnn"][0], atol=5e-4), "differs from the demo: check the data"
assert np.allclose(axis_acc["mlp"], demo["axis"]["mlp"], atol=5e-4), "differs from the demo: check the data"

The shortcut gives the same result bit for bit, and the accuracy of both networks matches the demo's file at every
shift. The figure below plots them together with two results that exist only in the file (they need long training):
CNN seeds 1 and 2 (blue vertical bars joining the lowest and the highest of the 3 seeds), and an MLP trained with
each image shifted randomly by up to ±2 pixels (data augmentation, Lesson 08).
MLP: hollow gray circles, dotted line; CNN: blue dots, solid line; MLP + shifts: orange squares, dashed line.

In [ ]:
ks = np.arange(5)
cnn_seeds = np.array(demo["axis"]["cnn"])  # 3 seeds × 5 shift levels (seed 0 was just recomputed above)
aug = np.array(demo["axis"]["mlpAug"])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(ks, 100 * np.array(axis_acc["mlp"]), color=GREY, ls=":", lw=2, marker="o", mfc="white", ms=8, label="MLP")
ax.vlines(ks, 100 * cnn_seeds.min(0), 100 * cnn_seeds.max(0), color=BLUE, lw=2)
ax.plot(ks, 100 * np.median(cnn_seeds, 0), color=BLUE, lw=2, marker="o", ms=7, label="CNN (median of 3 seeds)")
ax.plot(ks, 100 * aug, color=ORANGE, ls="--", lw=2, marker="s", ms=7, label="MLP trained on images shifted ±2")
ax.axhline(10, color="#555", lw=1, ls="--")
ax.text(4, 12, "random guessing 10%", ha="right", fontsize=9, color="#555")
ax.set(
    xlabel="pixels shifted (average of 4 directions)",
    ylabel="correct on 10,000 test images (%)",
    ylim=(0, 100),
    xticks=ks,
)
ax.set_title("Shifted test images: the CNN is more robust than the MLP, not invariant")
ax.legend(frameon=False, loc="lower left")
ax.grid(axis="y", alpha=0.3)
fmt_axes(fig)
plt.show()
for k in ks:
    print(
        f"shift {k}: MLP {pct(axis_acc['mlp'][k])} · CNN {pct(cnn_seeds[:, k].min())}–{pct(cnn_seeds[:, k].max())} (3 seeds)"
        f" · MLP + shifts ±2 {pct(aug[k])}"
    )

Shifted by 1 pixel, both networks lose only a few points (CNN 95.40–96.27% across the 3 seeds, MLP 94.39%). Shifted
by 4 pixels, the CNN drops to 36.78–42.46%: more robust than the MLP (29.49%), but far from invariant. The MLP trained
on randomly shifted images, on the other hand, keeps 75.33% at 4 pixels, better than every CNN seed from a shift of 1
onward. Robustness to shifts can be **learned from data**; the CNN has part of it built into its structure, with
8 times fewer parameters. At level 0, MLP + shifts (96.85%) is below all 3 CNN seeds (97.40–97.98%), so no network
wins at every level. (Each MLP curve is a single run, seed 0.)

### 🤔 Predict first
Shifted by 4 pixels, CNN seed 0 gets only 41.09% of the images right. Is that mainly because part of the stroke is
pushed out of the 28×28 frame? If we keep only the images that **lose no ink at all** when shifted (the 4 rows or
4 columns on the side that gets pushed out are all empty), does the accuracy come back to near 98%?

In [ ]:
ink = test_img > 0
edge = {(4, 0): ink[:, :, 24:], (-4, 0): ink[:, :, :4], (0, 4): ink[:, 24:, :], (0, -4): ink[:, :4, :]}
names = {(4, 0): "right", (-4, 0): "left", (0, 4): "down", (0, -4): "up"}
intact_acc = {}
for d, strip in edge.items():
    lost = strip.any(axis=(1, 2))  # images with ink in the strip that gets pushed out
    ok = right["cnn"][d]
    intact_acc[d] = ok[~lost].mean()
    print(
        f"shift 4 {names[d]:>5}: {pct(lost.mean()):>6} of images lose ink · CNN correct on {pct(ok.mean())} of all images, "
        f"on {pct(intact_acc[d])} of the {num((~lost).sum(), 0)} images that keep all their ink"
    )
assert max(intact_acc.values()) < 0.6

No. Even when no ink at all is lost, the CNN gets only 30.95% to 51.01% right, depending on the direction. Losing ink
does hurt when shifting up or down (the accuracy on the images that keep all their ink is higher than on all images),
but when shifting left or right, the images that lose ink are actually predicted correctly more often. The main
reason is **position**: each MNIST digit was shifted so that the center of mass of its stroke sits exactly at the
center of the 28×28 frame ([the original MNIST page](https://web.archive.org/web/20231215034634/http://yann.lecun.com/exdb/mnist/)),
so the dense layer has only ever seen strokes around that position. Shifted by 4 pixels, the feature maps still move
along exactly as before, but the signal lands on weights of the dense layer that have hardly ever been used.

The demo lets you shift each sample image and see what both networks answer. The cell below recomputes those answers
for all 81 shifts in $[-4, 4]^2$: the MLP's prediction, its probability, and how many of the 128 neurons of the first
layer switch their on/off state compared with the original image (the demo's file stores these three numbers), then
prints the results at the shift (+4, 0).

In [ ]:
mism, wrong4, flips4 = 0, {"MLP": 0, "CNN": 0}, []
for s, i in zip(demo["samples"], SAMPLES):
    moved = np.stack([shift(Xte[i : i + 1], a, b)[0] for a, b in SHIFTS])  # 81 images
    h1, _, z = mlp_forward(MLP, moved)
    p_mlp, p_cnn = softmax(z), softmax(cnn_logits(CNN, moved))
    flips = ((h1 > 0) != (h1[CENTER] > 0)).sum(1)
    for k, (pred, p, fl) in enumerate(s["mlp"]):
        mism += pred != p_mlp[k].argmax() or abs(p - p_mlp[k].max()) > 2e-3 or fl != flips[k]
    k = SHIFTS.index((4, 0))
    wrong4["MLP"] += p_mlp[k].argmax() != yte[i]
    wrong4["CNN"] += p_cnn[k].argmax() != yte[i]
    flips4.append(flips[k])
    print(
        f"#{i:>4} (digit {yte[i]}) shifted (+4, 0): MLP predicts {p_mlp[k].argmax()} ({pct(p_mlp[k].max(), 1)}, {flips[k]}/128 neurons switch)"
        f" · CNN predicts {p_cnn[k].argmax()} ({pct(p_cnn[k].max(), 1)})"
    )
print(
    f"shift (+4, 0): MLP wrong on {wrong4['MLP']}/10, CNN wrong on {wrong4['CNN']}/10; the MLP switches {min(flips4)}–{max(flips4)}/128 neurons"
)
print(f"differences from the demo's file: {mism} out of {len(SAMPLES) * len(SHIFTS)} (image, shift) pairs")
assert mism == 0, "differs from the demo: check the data"

At the shift (+4, 0), the MLP gets 9 of the 10 sample images wrong, and 29 to 74 of the 128 neurons in its first
layer switch state; the CNN also gets 7 images wrong. Many wrong answers still come with a high probability (the CNN
predicts the 7 as a 5 with 96.2%): the network doesn't know it is looking at an unusual image. These are images with
wide margins, so none of them loses ink when shifted.

## 11 · Shuffling the pixel positions

What makes the CNN better? A neat test: pick **one** fixed permutation of the 784 positions (seed 123, as in
`cnn.py`), shuffle every training and test image with that very permutation, then retrain. After shuffling, the
images look like noise (figure below), but no information is lost: the inverse permutation gives back the original
image.

In [ ]:
PERM = np.random.default_rng(123).permutation(784)
Xtr_perm = Xtr.reshape(-1, 784)[:, PERM].reshape(-1, 28, 28)
Xte_perm = Xte.reshape(-1, 784)[:, PERM].reshape(-1, 28, 28)
fig, axes = plt.subplots(2, 5, figsize=(8, 3.4))
for col, i in enumerate(SAMPLES[:5]):
    for row, (X, name) in enumerate([(Xte, "original"), (Xte_perm, "shuffled")]):
        axes[row, col].imshow(X[i], cmap="gray_r", vmin=0, vmax=1)
        axes[row, col].set_title(f"digit {yte[i]}, {name}", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()
assert np.array_equal(Xte_perm.reshape(-1, 784)[:, np.argsort(PERM)], Xte.reshape(-1, 784))

### 🤔 Predict first
Retrain the MLP and the CNN on the shuffled images, with the same recipe. Which network loses more accuracy? Does
either network lose nothing at all?

For the MLP, we can answer exactly. The MLP computes $xW_1$: shuffling the elements of $x$ is equivalent to just
**reordering the rows** of $W_1$. If the initial $W_1$ also has its rows shuffled with the same permutation, the two
networks must be identical at every step. The cell below checks this on one epoch over 10,000 images, then retrains
the CNN on the shuffled images (313 steps, as in section 7).

In [ ]:
Xf, Xf_perm = Xtr.reshape(-1, 784)[:10_000], Xtr_perm.reshape(-1, 784)[:10_000]
mlp_a = train_mlp(Xf, ytr[:10_000], epochs=1)
mlp_b = train_mlp(Xf_perm, ytr[:10_000], epochs=1, perm=PERM)
za, zb = mlp_forward(mlp_a, Xte)[2], mlp_forward(mlp_b, Xte_perm)[2]
print(
    f"MLP 1 epoch, 10,000 images: original {pct((za.argmax(1) == yte).mean())}, shuffled {pct((zb.argmax(1) == yte).mean())};"
    f" the logits differ by at most {sci(np.abs(za - zb).max())}"
)
assert np.array_equal(za.argmax(1), zb.argmax(1)) and np.allclose(za, zb, atol=1e-10)

P_perm, _ = train_cnn(seed=0, X=Xtr_perm)
acc_perm = accuracy(P_perm, Xte_perm)
print(f"CNN 313 steps, seed 0: original {pct(acc_live)}, shuffled {pct(acc_perm)}")
print(
    f"full recipe (the demo's file, seed 0): MLP {pct(demo['grid']['mlp'][CENTER])} → {pct(demo['perm']['mlp'])}, "
    f"CNN {pct(demo['grid']['cnn'][CENTER])} → {pct(demo['perm']['cnn'])}"
)
print(f"CNN on the original images across 3 seeds: {pct(cnn_seeds[:, 0].min())}–{pct(cnn_seeds[:, 0].max())}")

For the MLP, shuffling the pixels **changes nothing**: the same 80.90%, with logits equal up to rounding error. The
MLP has no idea which pixel sits next to which, so it loses nothing when that order is destroyed. In `cnn.py`, the MLP
on shuffled images is initialized with a $W_1$ whose rows are *not* shuffled, i.e. a different random initialization,
so it gets 97.50% instead of 97.27%: that difference comes only from the initialization.

The CNN, on the other hand, loses: 92.66% → 90.33% after 313 steps, and 97.98% → 95.09% with the full recipe.
95.09% lies far below the 3 seeds' range of 97.40–97.98%, so this is not variation between seeds. On shuffled images,
a $3 \times 3$ window gathers 9 pixels scattered all over the digit, so “local patterns” no longer mean anything, and
the CNN does even worse than the MLP. The CNN's advantage comes from an **assumption** about the data (an *inductive
bias*): pixels that are close to each other are related, and a pattern that is useful in one place is also useful in
other places. Real images satisfy that assumption; shuffled images don't.

## 12 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again.

### Do the filters need to be learned?
Section 5 showed that the 8 learned filters are still close to where they were at initialization. So what if we
**keep** the random filters fixed and learn only the dense layer? `learn_filters=False` does exactly that, with the
same seed (so the same initial filters) and the same 313 steps.

In [ ]:
SEEDS = (0, 1, 2)
res = {"learned": [], "fixed": []}
for seed in SEEDS:
    P_learn = P_live if seed == 0 else train_cnn(seed=seed)[0]  # seed 0 was trained in section 7
    res["learned"].append(accuracy(P_learn))
    res["fixed"].append(accuracy(train_cnn(seed=seed, learn_filters=False)[0]))
    print(f"seed {seed}: learned filters {pct(res['learned'][-1])} · fixed random filters {pct(res['fixed'][-1])}")
for name, v in res.items():
    print(f"{name:>8}: {pct(min(v))}–{pct(max(v))}")
gap = 100 * (np.array(res["learned"]) - np.array(res["fixed"]))
print(
    "learning the filters minus keeping them fixed (percentage points, per seed):", " · ".join(num(g, 2) for g in gap)
)

In all 3 seeds, learning the filters beats keeping them fixed, but only by 0.56 to 0.99 points. That difference is
smaller than the variation between seeds (90.45–92.66% with learned filters), and the two 3-seed ranges overlap; we
only see it because we compare pairs with the same seed (same initial filters, same image order). Most of the benefit
comes from the **architecture**: local windows, shared weights, pooling. 8 random filters already look in
8 different directions in the 9-dimensional space of $3 \times 3$ patterns, and the dense layer learns how to combine
them. Saxe et al. (2011) analyze why convolution + pooling architectures with random weights still give surprisingly
good features for recognition. This conclusion holds only for this small one-layer network: the filters in the deep
layers of a large CNN combine many features together, and they have to be learned.

### Your own filter, and the conditions for equivariance
Change `K_MINE` to any $3 \times 3$ filter. The cell below draws its map on the digit 4, then checks equivariance
with 1,000 random filters and two ways of sliding: stride 1 (as above) and **stride 2** (keeping only every second
cell in each direction, as some networks do to shrink the map).

In [ ]:
K_MINE = np.array(
    [[1, 1, 1], [0, 0, 0], [-1, -1, -1]], float
)  # try editing: a horizontal edge, “top row minus bottom row”
fig, (a, b) = plt.subplots(1, 2, figsize=(6.4, 3.2))
a.imshow(I, cmap="gray_r", vmin=0, vmax=1)
a.set_title("digit 4 (#109)", fontsize=10)
a.axis("off")
im = show_map(b, conv2d(I, K_MINE.reshape(9, 1))[..., 0], "map of K_MINE")
fig.colorbar(im, ax=b, fraction=0.046)
fmt_axes(fig)
plt.show()

W_rand = np.random.default_rng(5).normal(size=(9, 1000))  # 1,000 random filters
x0, x1, x2 = Xte[109:110], shift(Xte[109:110], 1, 0), shift(Xte[109:110], 2, 0)
Z0, Z1, Z2 = (conv2d(x, W_rand) for x in (x0, x1, x2))
print("stride 1, shift by 1 pixel: equivariant?", np.allclose(Z1, move_map(Z0, 1, 0, 0.0), atol=1e-12))
S0, S1, S2 = Z0[:, ::2, ::2], Z1[:, ::2, ::2], Z2[:, ::2, ::2]  # stride 2: 13 × 13 maps
print(
    "stride 2, shift by 2 pixels: does the map move by exactly 1 cell?",
    np.allclose(S2[:, :, 1:], S0[:, :, :-1], atol=1e-12),
)
rel = np.linalg.norm(S1[:, :, 1:] - S0[:, :, :-1]) / np.linalg.norm(S0[:, :, :-1])
print(
    f"stride 2, shift by 1 pixel: differs from the map shifted by 1 cell by {pct(rel, 0)}, from the unshifted map by "
    f"{pct(np.linalg.norm(S1 - S0) / np.linalg.norm(S0), 0)}"
)

With stride 1, every filter is equivariant: this property comes from the structure of the computation, not from the
weights. With stride 2, only **even** shifts are preserved; a 1-pixel shift gives a map that matches the old one
neither shifted by 0 nor by 1 cell, like max pooling in section 9. This is why Azulay & Weiss (2018) and Zhang (2019)
found that modern CNNs can change their answer when the image shifts by just 1 pixel: every downsampling step
(pooling, stride > 1) breaks equivariance for shifts that are not a multiple of the step.

## 13 · Limits and the bridge: the receptive field

Each cell of the map sees only a $3 \times 3$ window: that is its **receptive field**, the part of the image that can
change the cell's value. After 2×2 max pooling, a cell sees $4 \times 4$ pixels. In the demo's network, only the
dense layer sees the whole image, and it is tightly tied to position. To get features that see a large region while
still sharing weights, we **stack** several convolutional and pooling layers: each layer looks at the maps of the
layer before, so the receptive field grows.

The cell below measures the receptive field experimentally. We pass 784 images, each with only **one** pixel equal
to 1, through a stack of layers (with filters of all 1s, so no numbers cancel each other out), then count the pixels
that make the center cell of the last map nonzero.

In [ ]:
def receptive_field(layers):
    """Side of the receptive field of the last map's center cell: each layer is 'conv' (3×3, stride 1) or 'pool'
    (2×2, stride 2)."""
    A = np.eye(784).reshape(784, 28, 28, 1)  # 784 images, each with one bright pixel
    for layer in layers:
        if layer == "conv":
            A = conv2d(A[..., 0], np.ones((9, 1)))
        else:  # an odd side drops its last row/column, as libraries do
            A = maxpool2(A[:, : A.shape[1] // 2 * 2, : A.shape[2] // 2 * 2])
    seen = A[:, A.shape[1] // 2, A.shape[2] // 2, 0].reshape(28, 28) > 0
    rows, cols = np.nonzero(seen)
    side = int(np.ptp(rows)) + 1
    assert seen.sum() == side * (int(np.ptp(cols)) + 1)  # the visible region is one solid rectangle
    return side


stacks = {
    "convolution only": ["conv"] * 5,
    "convolution + pooling": ["conv", "pool", "conv", "pool", "conv"],
}
for name, layers in stacks.items():
    sizes = [receptive_field(layers[: n + 1]) for n in range(len(layers))]
    print(f"{name:>22}: " + " → ".join(f"{L} {s}×{s}" for L, s in zip(layers, sizes)))
assert receptive_field(["conv", "pool"]) == 4 and receptive_field(stacks["convolution + pooling"]) == 18

Before the first pooling layer, each 3×3 convolutional layer widens the receptive field by 2 pixels. Each 2×2 pooling
layer doubles the jump (the distance, in pixels of the input image, between two neighboring cells of a map), so a
3×3 convolutional layer after one pooling layer widens it by 4 pixels (4×4 → 8×8 above), and after two pooling layers
by 8 (10×10 → 18×18). Five convolutional layers in a row only reach 11×11; with pooling in between, the same 5 layers
already see 18×18, almost the whole digit. In large CNNs trained on ImageNet, Zeiler & Fergus (2013) observed that
the first layer responds to edges and colors, layer 2 to corners and edge–color combinations, layer 3 to textures,
and the later layers to parts and then whole objects. A receptive field that grows with depth is a precondition for
that hierarchy; the one-layer network of this lesson doesn't have it yet.

The network of this lesson has two more limits that we measured:
- **Not yet invariant:** shifted by 4 pixels, the CNN drops to 36.78–42.46% (3 seeds), because the dense layer behind
  it is tied to position.
- **Pooling isn't free:** max pooling is equivariant to even shifts but not to odd shifts.

[Lesson 12 · Pooling & classic CNN architectures](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/kien-truc-cnn/)
stacks convolution and pooling into LeNet, compares the kinds of pooling, and reruns this very shift test.

## Summary
- Convolution slides a $k \times k$ filter over the image: each cell is a sum of $k^2$ products plus $b$. A 28×28
  image and a 3×3 filter give a 26×26 map (6,084 MACs). Rewritten with im2col, it is a single matrix multiplication
  $Z = PW + b$.
- Deep learning libraries compute cross-correlation (without flipping the filter) and call it convolution; with
  learned filters, the two differ only by a flip.
- The vertical/horizontal Sobel filters light up at the edges of strokes. The demo's CNN learns its 8 filters by
  itself; 4 of them look like edge detectors (|cos| ≥ 0.75), even though each one stays close to its own direction
  at initialization (18° to 45° apart).
- CNN: 13,610 parameters, 97.98% correct (MLP: 109,386 parameters, 97.27%). The 80 convolution parameters cost
  48,672 MACs, 3.6 times as many as the dense layer, because they are reused at 676 positions.
- Convolution is exactly equivariant to shifts by whole pixels (676/676 cells; 800/800 image–shift pairs). Max
  pooling and stride 2 keep that only for even shifts; the dense layer doesn't keep it at all.
- Shifted by 4 pixels: MLP 29.49%, CNN 36.78–42.46%, and the MLP trained on images shifted ±2 gets 75.33%.
  Equivariance is not yet invariance.
- Shuffled pixels: the MLP doesn't change (it is just a permutation of the rows of $W_1$), the CNN drops from 97.98%
  to 95.09%. The CNN's advantage is the assumption that nearby pixels are related.

## Review questions
1. A 28×28 image, a 5×5 filter, stride 1, no padding: what size is the feature map, and how many MACs does it cost
   per filter?
2. A 3×3 filter has all its weights equal (1/9). What does it do to the image, and what does its map look like at
   the edge of a vertical stroke?
3. Why does the convolutional layer have more than 100 times fewer parameters than the dense layer, yet cost more
   operations?
4. Shift the image 2 pixels to the right. How does the map after convolution change? After 2×2 max pooling? And for
   a 1-pixel shift?
5. Why does shuffling the pixel positions (the same way for every image) not affect the MLP at all?
6. A network made only of 3×3 convolutional layers (stride 1, no pooling, zero padding at the border so the maps stay
   28×28) needs how many layers for its center cell to have a receptive field at least 28×28 wide?

<details><summary>Hints and answers</summary>

1. $(28 - 5 + 1)^2 = 24 \times 24 = 576$ cells, 25 MACs each: $576 \cdot 25 = 14{,}400$ MACs.
2. It averages 9 pixels, i.e. it blurs the image. At the edge of a stroke, the value goes gradually from 0 to the
   stroke's value instead of jumping; the map doesn't change sign the way the Sobel filter's does, because all the
   weights are positive.
3. Each convolution weight is reused at 676 positions (section 6): the parameter count counts distinct weights, the
   MAC count counts every use. The dense layer uses each weight exactly once per image.
4. After convolution: the map moves by exactly 2 cells (equivariance, as long as no stroke is pushed out of the
   frame). After 2×2 pooling: by exactly 1 cell. A 1-pixel shift: the convolution map moves by 1 cell, but after
   pooling the 2×2 blocks are cut differently, so the small map equals no shifted version of the old one.
5. If $x$ is shuffled, $xW_1$ is equivalent to just reordering the rows of $W_1$; the MLP can learn that $W_1$ like
   any other $W_1$ (section 11 checks this: an initialization with the corresponding rows reordered gives exactly the
   same network).
6. Each layer widens it by 2 pixels: $1 + 2L \ge 28$, so it takes $L = 14$ layers (a 29×29 region). Without padding,
   the map would already shrink to 2×2 after 13 layers. With pooling in between, far fewer layers are needed
   (section 13).
</details>

## Further reading
- Dive into Deep Learning, [*Convolutions for Images*](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html):
  the `corr2d` function as a loop, edge filters, learning a filter from data, cross-correlation and receptive fields.
- CS231n, [*Convolutional Neural Networks*](https://cs231n.github.io/convolutional-networks/): weight sharing
  (the AlexNet example), im2col, the formula $(W - F + 2P)/S + 1$.
- Goodfellow, Bengio & Courville, [*Deep Learning*, Chapter 9](https://www.deeplearningbook.org/contents/convnets.html):
  sparse interactions, parameter sharing, equivariance, pooling.
- V. Dumoulin & F. Visin (2016), [A guide to convolution arithmetic for deep learning](https://arxiv.org/abs/1603.07285).
- K. Fukushima (1980), [Neocognitron](https://doi.org/10.1007/BF00344251); Y. LeCun et al. (1989),
  [Backpropagation applied to handwritten zip code recognition](https://doi.org/10.1162/neco.1989.1.4.541) and (1998),
  [Gradient-based learning applied to document recognition](https://doi.org/10.1109/5.726791).
- M. Zeiler & R. Fergus (2013), [Visualizing and Understanding Convolutional Networks](https://arxiv.org/abs/1311.2901);
  C. Olah et al. (2017), [Feature Visualization](https://distill.pub/2017/feature-visualization/).
- A. Saxe et al. (2011), [On random weights and unsupervised feature learning](http://www.icml-2011.org/papers/551_icmlpaper.pdf):
  why convolution + pooling architectures with random weights still give good features.
- A. Azulay & Y. Weiss (2018), [Why do deep convolutional networks generalize so poorly to small image transformations?](https://arxiv.org/abs/1805.12177);
  R. Zhang (2019), [Making Convolutional Networks Shift-Invariant Again](https://arxiv.org/abs/1904.11486);
  O. Kayhan & J. van Gemert (2020), [On Translation Invariance in CNNs](https://arxiv.org/abs/2003.07064).
- Interactive tools: [CNN Explainer](https://poloclub.github.io/cnn-explainer/), [Image Kernels](https://setosa.io/ev/image-kernels/).

**Next lesson:** [Lesson 12 · Pooling & classic CNN architectures](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/kien-truc-cnn/):
stack convolution and pooling into LeNet, and measure how well each way of shrinking the map preserves equivariance.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")